##Loading Libraries

In [ ]:
##!pip install datasets transformers accelerate

In [ ]:
import pandas as pd
import re
from datasets import Dataset
from transformers import AutoTokenizer
from sklearn.model_selection import train_test_split
import torch
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score

## Loading Data and Merging

In [ ]:
# Parse the Tweets File
with open('Sarcasm_tweets.txt', 'r', encoding='utf-8') as f:
    tweet_lines = f.read().splitlines()

In [ ]:
tweets_data = []
# The file structure seems to be: ID, Text, Blank Line
for i in range(0, len(tweet_lines), 3):
    if i+1 < len(tweet_lines):
        tweet_id = tweet_lines[i].strip()
        text = tweet_lines[i+1].strip()
        tweets_data.append({'ID': tweet_id, 'Text': text})

In [ ]:
df_tweets = pd.DataFrame(tweets_data)

In [ ]:
df_tweets.head()

,ID,Text
0,866871160725794816,Triple Talaq par Burbak Kuchh nahi bolega
1,880356789358743553,Batao ye uss site pr se akki sir ke verdict ni...
2,877751493889105920,Hindu baheno par julam bardas nahi hoga @Tripl...
3,901806457871466496,Naa bhai.. aisa nhi hai.. mere handle karne se...
4,866264330748219392,#RememberingRajiv aaj agar musalman auraten tr...


In [ ]:
df_tweets.shape

(5250, 2)

In [ ]:
# Parse the Truth (Labels) File
with open('Sarcasm_tweet_truth.txt', 'r', encoding='utf-8') as f:
    truth_lines = f.read().splitlines()

In [ ]:
truth_data = []
# The file structure seems to be: ID, YES/NO
for i in range(0, len(truth_lines), 2):
    if i+1 < len(truth_lines):
        tweet_id = truth_lines[i].strip()
        label = truth_lines[i+1].strip()
        truth_data.append({'ID': tweet_id, 'Label': label})

In [ ]:
df_truth = pd.DataFrame(truth_data)

In [ ]:
df_truth.head()

,ID,Label
0,866871160725794816,NO
1,880356789358743553,YES
2,877751493889105920,NO
3,901806457871466496,NO
4,866264330748219392,NO


In [ ]:
df_truth.shape

(5250, 2)

In [ ]:
df = pd.merge(df_tweets, df_truth, on='ID')

In [ ]:
# Convert labels to binary (1 for Sarcastic, 0 for Non-Sarcastic)
df['Label'] = df['Label'].map({'YES': 1, 'NO': 0})

In [ ]:
print(f"Total merged records: {len(df)}")
df.head()

Total merged records: 5250


,ID,Text,Label
0,866871160725794816,Triple Talaq par Burbak Kuchh nahi bolega,0
1,880356789358743553,Batao ye uss site pr se akki sir ke verdict ni...,1
2,877751493889105920,Hindu baheno par julam bardas nahi hoga @Tripl...,0
3,901806457871466496,Naa bhai.. aisa nhi hai.. mere handle karne se...,0
4,866264330748219392,#RememberingRajiv aaj agar musalman auraten tr...,0


In [ ]:
df.shape

(5250, 3)

## Data Cleaning

In [ ]:
import pandas as pd
import re

def preprocess_hinglish_tweet(text):
    # Safety check for missing values
    if not isinstance(text, str):
        return ""

    # 1. Remove URLs (The re.search line has been removed!)
    text = re.sub(r'http\S+|www\.\S+', '', text)

    # 2. Remove User Mentions (e.g., @username)
    text = re.sub(r'@\w+', '', text)

    # 3. Handle Hashtags (Remove the '#' symbol but keep the word)
    text = re.sub(r'#', '', text)

    # 4. Remove special characters but KEEP standard punctuation (! ? , .)
    text = re.sub(r'[^A-Za-z0-9\s!?.,\u0900-\u097F]', '', text)

    # 5. Condense multiple spaces into a single space
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# Apply the fixed function
df['Cleaned_Text'] = df['Text'].apply(preprocess_hinglish_tweet)

# Verify the output
df[['Text', 'Cleaned_Text']].head(10)

,Text,Cleaned_Text
0,Triple Talaq par Burbak Kuchh nahi bolega,Triple Talaq par Burbak Kuchh nahi bolega
1,Batao ye uss site pr se akki sir ke verdict ni...,Batao ye uss site pr se akki sir ke verdict ni...
2,Hindu baheno par julam bardas nahi hoga @Tripl...,Hindu baheno par julam bardas nahi hoga Hindu ...
3,Naa bhai.. aisa nhi hai.. mere handle karne se...,Naa bhai.. aisa nhi hai.. mere handle karne se...
4,#RememberingRajiv aaj agar musalman auraten tr...,RememberingRajiv aaj agar musalman auraten tri...
5,are cricket se sanyas le liya kya viru aur soc...,are cricket se sanyas le liya kya viru aur soc...
6,Sohail bahi hame bhi treen kardo muje bhi shoc...,Sohail bahi hame bhi treen kardo muje bhi shoc...
7,bhai triple talaq se aap kya samjhte hai samjh...,bhai triple talaq se aap kya samjhte hai samjh...
8,Bachcho ki death par politics ke bajay unke li...,Bachcho ki death par politics ke bajay unke li...
9,#@spn_cricket lijiye S Thakur ne phir pahni no...,"lijiye S Thakur ne phir pahni no.10 jarsi, ink..."


In [ ]:
df.head()

,ID,Text,Label,Cleaned_Text
0,866871160725794816,Triple Talaq par Burbak Kuchh nahi bolega,0,Triple Talaq par Burbak Kuchh nahi bolega
1,880356789358743553,Batao ye uss site pr se akki sir ke verdict ni...,1,Batao ye uss site pr se akki sir ke verdict ni...
2,877751493889105920,Hindu baheno par julam bardas nahi hoga @Tripl...,0,Hindu baheno par julam bardas nahi hoga Hindu ...
3,901806457871466496,Naa bhai.. aisa nhi hai.. mere handle karne se...,0,Naa bhai.. aisa nhi hai.. mere handle karne se...
4,866264330748219392,#RememberingRajiv aaj agar musalman auraten tr...,0,RememberingRajiv aaj agar musalman auraten tri...


In [ ]:
df['Label'].value_counts()

,count
Label,
0,4746
1,504


## Splitting the dataset into training, valication & testing

In [ ]:
# First Split: Allocate 80% to Training and 20% to a temporary holding dataframe
train_df, temp_df = train_test_split(
    df,
    train_size=0.8,        # 80% of the dataset
    stratify=df['Label'],
    random_state=42
)

# Second Split: Divide that remaining 20% evenly (50/50) to get your two 10% sets
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,         # 50% of the remaining 20% equals exactly 10% of the total dataset
    stratify=temp_df['Label'],
    random_state=42
)

print(f"Dataset Split Complete:")
print(f"Training Set (80%): {len(train_df)} instances")
print(f"Validation Set (10%): {len(val_df)} instances")
print(f"Testing Set (10%): {len(test_df)} instances")

Dataset Split Complete:
Training Set (80%): 4200 instances
Validation Set (10%): 525 instances
Testing Set (10%): 525 instances


## Tokenization Process

In [ ]:
# Convert DataFrames to Hugging Face format
train_ds = Dataset.from_pandas(train_df[['Cleaned_Text', 'Label']].reset_index(drop=True))
val_ds = Dataset.from_pandas(val_df[['Cleaned_Text', 'Label']].reset_index(drop=True))
test_ds = Dataset.from_pandas(test_df[['Cleaned_Text', 'Label']].reset_index(drop=True))

# Load the MuRIL Tokenizer
tokenizer = AutoTokenizer.from_pretrained("google/muril-base-cased")

# Tokenize the text (Pad to 128 tokens, truncate if longer)
def tokenize(batch):
    return tokenizer(batch["Cleaned_Text"], padding="max_length", truncation=True, max_length=128)

# Apply mapping and format to PyTorch Tensors
tokenized_train = train_ds.map(tokenize, batched=True).remove_columns(["Cleaned_Text"]).rename_column("Label", "labels")
tokenized_val = val_ds.map(tokenize, batched=True).remove_columns(["Cleaned_Text"]).rename_column("Label", "labels")
tokenized_test = test_ds.map(tokenize, batched=True).remove_columns(["Cleaned_Text"]).rename_column("Label", "labels")

tokenized_train.set_format("torch")
tokenized_val.set_format("torch")
tokenized_test.set_format("torch")

print("Tokenization complete! Ready for the model.")

Map:   0%|          | 0/4200 [00:00<?, ? examples/s]

Map:   0%|          | 0/525 [00:00<?, ? examples/s]

Map:   0%|          | 0/525 [00:00<?, ? examples/s]

Tokenization complete! Ready for the model.


## Training the model

In [ ]:
import torch
from torch import nn
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from transformers import Trainer, AutoModelForSequenceClassification, TrainingArguments

# 1. Calculate the weights based on your training data distribution
counts = train_df['Label'].value_counts()
w0 = len(train_df) / (2 * counts[0])
w1 = len(train_df) / (2 * counts[1])

print(f"Applying Class Weights -> Non-Sarcastic: {w0:.4f}, Sarcastic: {w1:.4f}")

# 2. Define the Custom Trainer that handles the imbalance
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        weights = torch.tensor([w0, w1], dtype=torch.float).to(model.device)
        loss_fct = nn.CrossEntropyLoss(weight=weights)

        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

# 3. Define the metrics function for evaluation
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, predictions),
        'f1_macro': f1_score(labels, predictions, average='macro')
    }

# 4. Reset the model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AutoModelForSequenceClassification.from_pretrained(
    "google/muril-base-cased",
    num_labels=2
).to(device)

# 5. Define Training Arguments (Warning fixed: removed 'logging_dir')
training_args = TrainingArguments(
    output_dir='./results',          # Output directory
    num_train_epochs=3,              # Total number of training epochs
    per_device_train_batch_size=16,  # Batch size per device during training
    per_device_eval_batch_size=16,   # Batch size for evaluation
    eval_strategy="epoch",           # Evaluation is done at the end of each epoch
    save_strategy="epoch",           # Save model checkpoint at the end of each epoch
    logging_steps=10,
)

# 6. Initialize the WeightedTrainer
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics  # This now points to the function we defined in step 3!
)

# 7. Start Retraining
print("Starting Weighted Fine-Tuning...")
trainer.train()

Applying Class Weights -> Non-Sarcastic: 0.5531, Sarcastic: 5.2109


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: google/muril-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expe

Starting Weighted Fine-Tuning...


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.163180,0.290081,0.954286,0.883704
2,0.197232,0.282199,0.956190,0.889384
3,0.094348,0.281769,0.956190,0.889384


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=789, training_loss=0.23407810624259204, metrics={'train_runtime': 707.6227, 'train_samples_per_second': 17.806, 'train_steps_per_second': 1.115, 'total_flos': 828799824384000.0, 'train_loss': 0.23407810624259204, 'epoch': 3.0})

## Model Evaluation

In [ ]:
# Evaluate on the Test Set
print("Running final evaluation on the Test Set...")
test_results = trainer.predict(tokenized_test)

# Display the numbers
print("\n--- TEST SET RESULTS ---")
print(f"Final Accuracy: {test_results.metrics['test_accuracy']:.4f}")
print(f"Final Macro F1: {test_results.metrics['test_f1_macro']:.4f}")

# Quick Check: Did the model only predict one class?
preds = test_results.predictions.argmax(-1)
print("\nPrediction Distribution:")
print(pd.Series(preds).value_counts())

Running final evaluation on the Test Set...



--- TEST SET RESULTS ---
Final Accuracy: 0.9486
Final Macro F1: 0.8739

Prediction Distribution:
0    454
1     71
Name: count, dtype: int64


In [ ]:
# Run final predictions on the 10% Test Set
print("Generating final test set predictions...")
test_results = trainer.predict(tokenized_test)

# Print the official metrics
print("\n" + "="*30)
print("FINAL TEST PERFORMANCE")
print("="*30)
print(f"Accuracy: {test_results.metrics['test_accuracy']:.4f}")
print(f"Macro F1: {test_results.metrics['test_f1_macro']:.4f}")
print("="*30)

# 3. Check the Prediction Distribution
preds = test_results.predictions.argmax(-1)
print("\nPrediction Distribution on Test Set:")
print(pd.Series(preds).value_counts())

Generating final test set predictions...



FINAL TEST PERFORMANCE
Accuracy: 0.9486
Macro F1: 0.8739

Prediction Distribution on Test Set:
0    454
1     71
Name: count, dtype: int64


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

# Generate the detailed Classification Report
report = classification_report(test_df['Label'], preds, target_names=['Non-Sarcastic', 'Sarcastic'])
print("DETAILED CLASSIFICATION REPORT:")
print(report)

# 2. Identify the "False Negatives" (Sarcasm the model missed)
test_df_analysis = test_df.copy()
test_df_analysis['Predicted'] = preds
missed_sarcasm = test_df_analysis[(test_df_analysis['Label'] == 1) & (test_df_analysis['Predicted'] == 0)]

print(f"\nThe model missed {len(missed_sarcasm)} sarcastic tweets.")
print("Example of a missed sarcastic tweet:")
print(missed_sarcasm['Cleaned_Text'].iloc[0] if not missed_sarcasm.empty else "No misses!")

DETAILED CLASSIFICATION REPORT:
               precision    recall  f1-score   support

Non-Sarcastic       0.99      0.95      0.97       475
    Sarcastic       0.66      0.94      0.78        50

     accuracy                           0.95       525
    macro avg       0.83      0.94      0.87       525
 weighted avg       0.96      0.95      0.95       525


The model missed 3 sarcastic tweets.
Example of a missed sarcastic tweet:
Kajol has most dusky looks in the song Yeh kaali kaali ankhen, Yeh Gore Gore Gaal of Bazigar.


In [ ]:
#  Calculate Training Accuracy (on the 4,170 training instances)
print("Calculating Accuracy on the Training Set (this may take a minute)...")
train_results = trainer.predict(tokenized_train)
train_acc = train_results.metrics['test_accuracy'] # 'test_accuracy' is just the label HF uses for predictions

# Re-confirm Test Accuracy
print("Confirming Accuracy on the Test Set...")
test_results = trainer.predict(tokenized_test)
test_acc = test_results.metrics['test_accuracy']

print("\n" + "="*30)
print("FINAL ACCURACY REPORT")
print("="*30)
print(f"Final Training Accuracy: {train_acc:.4f} ({train_acc*100:.2f}%)")
print(f"Final Test Accuracy:     {test_acc:.4f} ({test_acc*100:.2f}%)")
print("="*30)

Calculating Accuracy on the Training Set (this may take a minute)...


Confirming Accuracy on the Test Set...



FINAL ACCURACY REPORT
Final Training Accuracy: 0.9507 (95.07%)
Final Test Accuracy:     0.9486 (94.86%)
